In [1]:
"""
CarbonTatva AI — Model: XGBoost + PCA + Residual Analysis
==========================================================
Market : GDAM only
Data   : Last-year (Mar 2025 – Feb 2026)
Split  : Train ≤ Sep-2025  |  Val ≤ Nov-2025  |  Test > Nov-2025
Target : price / MA7d ratio  (handles intra-year level shifts)

Changes vs previous version:
  1. MARKETS = ['GDAM'] only
  2. PCA applied after StandardScaler — variance threshold 0.995
     (keeps ~95% of components, removes collinear lag/MA features)
  3. Residual analysis block added at end:
     - hourly bias plot (over/under by time of day)
     - price-bucket bias (low / mid / high / spike regimes)
     - weekday vs weekend residuals
     - residual autocorrelation (lag 1-48 slots)
     - top-10 worst-predicted days
"""

import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
OUT = 'outputs_xgboost_pca'
os.makedirs(OUT, exist_ok=True)

# ─────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────
DATA_FILE       = 'final_merged_data.xlsx'
MARKETS         = ['GDAM']          # GDAM only
SLOTS_DAY       = 96
LAST_YEAR_START = '2025-03-01'
TRAIN_END       = '2025-09-30'
VAL_END         = '2025-11-30'
PCA_VAR         = 0.995             # keep components that explain 99.5% variance
SEED            = 42
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING — build on FULL data first
# ─────────────────────────────────────────────────────────────────
def build_features(df):
    df = df.copy()
    d = pd.to_datetime(df['Delivery Date'], dayfirst=True)
    t = df['Time Period'].str.split('-').str[0]
    df['datetime'] = pd.to_datetime(d.astype(str) + ' ' + t)
    df = df.sort_values('datetime').reset_index(drop=True)

    df.drop(columns=['apparent_temperature (°C)',
                     'sunshine_duration (s)',
                     'precipitation (mm)'], errors='ignore', inplace=True)

    df['hour']    = df['datetime'].dt.hour
    df['dow']     = df['datetime'].dt.dayofweek
    df['month']   = df['datetime'].dt.month
    df['quarter'] = df['datetime'].dt.quarter

    df['hour_sin']  = np.sin(2 * np.pi * df['hour']  / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']  / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']   = np.sin(2 * np.pi * df['dow']   / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['dow']   / 7)
    df['is_weekend']   = (df['dow'] >= 5).astype(int)
    df['is_morning']   = ((df['hour'] >= 6)  & (df['hour'] <= 9)).astype(int)
    df['is_evening']   = ((df['hour'] >= 18) & (df['hour'] <= 21)).astype(int)
    df['is_solar_hrs'] = ((df['hour'] >= 8)  & (df['hour'] <= 16)).astype(int)

    try:
        import holidays
        india_h = holidays.India(years=range(2022, 2027))
        df['holiday_flag'] = df['datetime'].dt.date.apply(
            lambda d: 1 if d in india_h else 0)
    except ImportError:
        df['holiday_flag'] = 0

    p = df['Price (Rs./MWh)']
    df['price_lag96']   = p.shift(96)
    df['price_lag192']  = p.shift(192)
    df['price_lag288']  = p.shift(288)
    df['price_lag672']  = p.shift(672)
    df['price_lag1344'] = p.shift(1344)
    df['ma1d']  = p.shift(1).rolling(96).mean()
    df['ma3d']  = p.shift(1).rolling(288).mean()
    df['ma7d']  = p.shift(1).rolling(672).mean()
    df['ma14d'] = p.shift(1).rolling(1344).mean()
    df['ma30d'] = p.shift(1).rolling(2880).mean()
    df['std3d'] = p.shift(1).rolling(288).std()
    df['std7d'] = p.shift(1).rolling(672).std()
    df['price_diff1d'] = p.shift(96)   - p.shift(192)
    df['price_diff1w'] = p.shift(672)  - p.shift(1344)
    df['price_diff2w'] = p.shift(1344) - p.shift(2016)
    df['max_prev_day']   = p.shift(1).rolling(96).max()
    df['min_prev_day']   = p.shift(1).rolling(96).min()
    df['range_prev_day'] = df['max_prev_day'] - df['min_prev_day']
    df['price_level_ratio_7d']  = df['price_lag96'] / (df['ma7d']  + 1e-6)
    df['price_level_ratio_30d'] = df['price_lag96'] / (df['ma30d'] + 1e-6)
    df['rel_lag96']  = df['price_lag96']  / (df['ma7d']  + 1e-6)
    df['rel_lag672'] = df['price_lag672'] / (df['ma30d'] + 1e-6)

    df['temp']    = df['temperature_2m (°C)']
    df['temp_sq'] = df['temp'] ** 2
    df['radiation']            = df['shortwave_radiation (W/m²)']
    df['radiation_x_hour_sin'] = df['radiation'] * df['hour_sin']
    df['radiation_x_is_solar'] = df['radiation'] * df['is_solar_hrs']
    df['wind_speed']  = df['wind_speed_100m (km/h)']
    df['cloud_cover'] = df['cloud_cover (%)']
    df['humidity']    = df['relative_humidity_2m (%)']
    df['rain']        = df['rain (mm)']
    df['temp_x_evening']    = df['temp'] * df['is_evening']
    df['temp_x_morning']    = df['temp'] * df['is_morning']
    df['temp_x_is_weekend'] = df['temp'] * df['is_weekend']

    df['target_ratio'] = p / (df['ma7d'] + 1e-6)
    return df


FEATURES = [
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'dow_sin', 'dow_cos', 'quarter',
    'is_weekend', 'holiday_flag',
    'is_morning', 'is_evening', 'is_solar_hrs',
    'price_lag96', 'price_lag192', 'price_lag288',
    'price_lag672', 'price_lag1344',
    'ma1d', 'ma3d', 'ma7d', 'ma14d', 'ma30d',
    'std3d', 'std7d',
    'price_diff1d', 'price_diff1w', 'price_diff2w',
    'max_prev_day', 'min_prev_day', 'range_prev_day',
    'price_level_ratio_7d', 'price_level_ratio_30d',
    'rel_lag96', 'rel_lag672',
    'temp', 'temp_sq',
    'radiation', 'radiation_x_hour_sin', 'radiation_x_is_solar',
    'wind_speed', 'cloud_cover', 'humidity', 'rain',
    'temp_x_evening', 'temp_x_morning', 'temp_x_is_weekend',
]

# ─────────────────────────────────────────────────────────────────
# UTILITIES
# ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, label=''):
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    pct  = mae / np.mean(y_true) * 100
    print(f"    {label:22s}  MAE={mae:7.1f}  RMSE={rmse:7.1f}  MAPE={mape:.2f}%  ({pct:.1f}% of mean)")
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def save_pred_plot(y_true, y_pred, title, path):
    fig, axes = plt.subplots(3, 1, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
    n = min(800, len(y_true))
    axes[0].plot(y_true[:n], color='#aaa', lw=0.8, label='Actual')
    axes[0].plot(y_pred[:n], color='#e67e22', lw=0.8, label='Predicted')
    axes[0].set_title(f'{title} — Test Set', color='white', fontsize=12)
    axes[0].legend(fontsize=9); axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    n2 = SLOTS_DAY * 5
    axes[1].plot(y_true[:n2], color='#aaa', lw=1.5, label='Actual')
    axes[1].plot(y_pred[:n2], color='#e67e22', lw=1.5, ls='--', label='Predicted')
    axes[1].set_title(f'{title} — First 5 Days', color='white', fontsize=12)
    axes[1].legend(fontsize=9); axes[1].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[2].scatter(y_true[:3000], y_pred[:3000], s=2, alpha=0.3, color='#e67e22')
    lim = max(y_true[:3000].max(), y_pred[:3000].max())
    axes[2].plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect')
    axes[2].set_title(f'{title} — Actual vs Predicted', color='white', fontsize=12)
    axes[2].set_xlabel('Actual', color='#aaa'); axes[2].set_ylabel('Predicted', color='#aaa')
    axes[2].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()


def save_monthly_plot(test_df, test_pred, mkt, path):
    tc = test_df.copy()
    tc['predicted'] = test_pred
    tc['abs_error'] = np.abs(tc['Price (Rs./MWh)'] - test_pred)
    tc['ym'] = tc['datetime'].dt.to_period('M')
    monthly = tc.groupby('ym').agg(
        actual=('Price (Rs./MWh)', 'mean'),
        predicted=('predicted', 'mean'),
        mae=('abs_error', 'mean')).round(1)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    fig.patch.set_facecolor('#0f1117')
    for ax in axes:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
    months = monthly.index.astype(str)
    x = np.arange(len(months)); w = 0.35
    axes[0].bar(x-w/2, monthly['actual'],    w, color='#aaa', alpha=0.8, label='Actual')
    axes[0].bar(x+w/2, monthly['predicted'], w, color='#e67e22', alpha=0.8, label='Predicted')
    axes[0].set_title(f'{mkt} — Monthly Actual vs Predicted', color='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(months, rotation=30, ha='right')
    axes[0].legend(fontsize=9); axes[0].set_ylabel('Price (Rs./MWh)', color='#aaa')
    axes[1].bar(x, monthly['mae'], color='#e74c3c', alpha=0.85)
    axes[1].set_title(f'{mkt} — Monthly MAE', color='white')
    axes[1].set_xticks(x); axes[1].set_xticklabels(months, rotation=30, ha='right')
    axes[1].set_ylabel('MAE', color='#aaa')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()
    print(f"\n  Monthly breakdown:\n{monthly.to_string()}")


# ─────────────────────────────────────────────────────────────────
# RESIDUAL ANALYSIS — GDAM specific deep-dive
# ─────────────────────────────────────────────────────────────────
def residual_analysis(test_df, y_true, y_pred, mkt, out_dir):
    """
    Produces 4 diagnostic plots:
      1. Mean residual (bias) by hour of day   → where model over/under estimates
      2. Mean residual by price bucket          → performance across price regimes
      3. Mean residual by weekday               → structural weekly bias
      4. Residual autocorrelation (lags 1-48)  → unmodelled serial structure
    Also prints top-10 worst days.
    """
    df = test_df.copy().reset_index(drop=True)
    df['residual'] = y_pred - y_true       # positive = overestimate, negative = underestimate
    df['abs_err']  = np.abs(df['residual'])
    df['hour']     = df['datetime'].dt.hour
    df['dow']      = df['datetime'].dt.dayofweek
    df['date']     = df['datetime'].dt.date

    # ── price buckets ─────────────────────────────────────────────
    q33, q67 = np.percentile(y_true, 33), np.percentile(y_true, 67)
    spike_thr = np.percentile(y_true, 95)
    def price_bucket(p):
        if p < q33:      return '① Low (<P33)'
        elif p < q67:    return '② Mid (P33-P67)'
        elif p < spike_thr: return '③ High (P67-P95)'
        else:            return '④ Spike (>P95)'
    df['bucket'] = y_true
    df['bucket'] = df['bucket'].apply(price_bucket)

    day_names = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.patch.set_facecolor('#0f1117')
    fig.suptitle(f'Residual Analysis — XGBoost GDAM\n'
                 f'(+ve = overestimate, –ve = underestimate)', color='white', fontsize=13)
    for ax in axes.flat:
        ax.set_facecolor('#1a1d27'); ax.tick_params(colors='#aaa')
        ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.5)

    # Plot 1 — hourly bias
    hrly = df.groupby('hour')['residual'].mean()
    colors_hr = ['#e74c3c' if v > 0 else '#2980b9' for v in hrly.values]
    axes[0,0].bar(hrly.index, hrly.values, color=colors_hr, alpha=0.85)
    axes[0,0].set_title('Bias by Hour of Day', color='white')
    axes[0,0].set_xlabel('Hour', color='#aaa')
    axes[0,0].set_ylabel('Mean Residual (Rs./MWh)', color='#aaa')
    axes[0,0].set_xticks(range(0, 24, 2))
    # annotate worst hours
    worst_h = hrly.abs().nlargest(3).index
    for h in worst_h:
        axes[0,0].annotate(f'{hrly[h]:+.0f}', xy=(h, hrly[h]),
                           xytext=(h, hrly[h] + (30 if hrly[h]>0 else -30)),
                           color='yellow', fontsize=8, ha='center')

    # Plot 2 — price bucket bias
    bkt_order = ['① Low (<P33)', '② Mid (P33-P67)', '③ High (P67-P95)', '④ Spike (>P95)']
    bkt_bias = df.groupby('bucket')['residual'].mean().reindex(bkt_order)
    bkt_mae  = df.groupby('bucket')['abs_err'].mean().reindex(bkt_order)
    x = np.arange(len(bkt_order))
    w = 0.4
    colors_bkt = ['#e74c3c' if v > 0 else '#2980b9' for v in bkt_bias.values]
    axes[0,1].bar(x - w/2, bkt_bias.values, w, color=colors_bkt, alpha=0.85, label='Bias')
    axes[0,1].bar(x + w/2, bkt_mae.values,  w, color='#f39c12',  alpha=0.7,  label='MAE')
    axes[0,1].set_title('Bias & MAE by Price Regime', color='white')
    axes[0,1].set_xticks(x)
    axes[0,1].set_xticklabels([b.split('(')[0].strip() for b in bkt_order],
                               rotation=15, ha='right', fontsize=9)
    axes[0,1].set_ylabel('Rs./MWh', color='#aaa')
    axes[0,1].legend(fontsize=9)
    for i, (b, m) in enumerate(zip(bkt_bias.values, bkt_mae.values)):
        axes[0,1].text(i-w/2, b + (15 if b>=0 else -25), f'{b:+.0f}', ha='center',
                       color='white', fontsize=8)
        axes[0,1].text(i+w/2, m+5, f'{m:.0f}', ha='center', color='white', fontsize=8)

    # Plot 3 — weekday bias
    dow_bias = df.groupby('dow')['residual'].mean()
    dow_mae  = df.groupby('dow')['abs_err'].mean()
    x3 = np.arange(7)
    colors_dow = ['#e74c3c' if v > 0 else '#2980b9' for v in dow_bias.values]
    axes[1,0].bar(x3 - w/2, dow_bias.values, w, color=colors_dow, alpha=0.85, label='Bias')
    axes[1,0].bar(x3 + w/2, dow_mae.values,  w, color='#f39c12',  alpha=0.7,  label='MAE')
    axes[1,0].set_title('Bias & MAE by Weekday', color='white')
    axes[1,0].set_xticks(x3); axes[1,0].set_xticklabels(day_names)
    axes[1,0].set_ylabel('Rs./MWh', color='#aaa')
    axes[1,0].legend(fontsize=9)
    # shade weekend
    axes[1,0].axvspan(4.5, 6.5, color='white', alpha=0.05)
    axes[1,0].text(5.5, axes[1,0].get_ylim()[1]*0.9, 'Weekend',
                   color='#aaa', ha='center', fontsize=8)

    # Plot 4 — residual autocorrelation
    res = df['residual'].values
    max_lag = 48
    acf_vals = [np.corrcoef(res[:-lag], res[lag:])[0,1] for lag in range(1, max_lag+1)]
    ci = 1.96 / np.sqrt(len(res))
    colors_acf = ['#e74c3c' if abs(v) > ci else '#555' for v in acf_vals]
    axes[1,1].bar(range(1, max_lag+1), acf_vals, color=colors_acf, alpha=0.85)
    axes[1,1].axhline( ci, color='yellow', lw=1, ls='--', alpha=0.7, label=f'95% CI (±{ci:.3f})')
    axes[1,1].axhline(-ci, color='yellow', lw=1, ls='--', alpha=0.7)
    axes[1,1].set_title('Residual Autocorrelation (Lags 1–48 slots)', color='white')
    axes[1,1].set_xlabel('Lag (15-min slots)', color='#aaa')
    axes[1,1].set_ylabel('Correlation', color='#aaa')
    axes[1,1].legend(fontsize=9)
    # annotate significant lags
    sig_lags = [i+1 for i, v in enumerate(acf_vals) if abs(v) > ci]
    if sig_lags:
        axes[1,1].text(max_lag*0.6, 0.8*max(acf_vals),
                       f'Significant lags: {sig_lags[:5]}{"…" if len(sig_lags)>5 else ""}',
                       color='yellow', fontsize=8)

    plt.tight_layout()
    plt.savefig(f'{out_dir}/residual_analysis_{mkt}.png',
                dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()
    print(f"  Saved: {out_dir}/residual_analysis_{mkt}.png")

    # ── worst 10 days ────────────────────────────────────────────
    day_mae = df.groupby('date')['abs_err'].mean().nlargest(10)
    print(f"\n  Top-10 worst predicted days (by MAE):")
    for dt, mae_val in day_mae.items():
        sub = df[df['date'] == dt]
        bias_val = sub['residual'].mean()
        direction = 'OVER' if bias_val > 0 else 'UNDER'
        print(f"    {dt}  MAE={mae_val:.0f}  avg_bias={bias_val:+.0f} ({direction})")

    # ── summary text ─────────────────────────────────────────────
    print(f"\n  ── Residual Summary ──────────────────────────────────")
    print(f"  Overall bias (mean residual): {df['residual'].mean():+.1f} Rs./MWh")
    print(f"  Most overestimated hour : {hrly.idxmax():02d}:00  ({hrly.max():+.0f})")
    print(f"  Most underestimated hour: {hrly.idxmin():02d}:00  ({hrly.min():+.0f})")
    print(f"  Highest MAE price regime: {bkt_mae.idxmax()}  (MAE={bkt_mae.max():.0f})")
    sig_pct = 100 * len(sig_lags) / max_lag
    print(f"  Autocorrelated lags     : {len(sig_lags)}/{max_lag} ({sig_pct:.0f}%) — "
          + ("serial structure remains" if len(sig_lags) > 5 else "mostly white noise ✓"))


# ─────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("MODEL: XGBoost + PCA (Last Year Data)")
print(f"Market: GDAM only")
print(f"Period: {LAST_YEAR_START} → Feb 2026")
print(f"Split:  Train≤{TRAIN_END}  |  Val≤{VAL_END}  |  Test>Val")
print(f"PCA variance threshold: {PCA_VAR}")
print("=" * 60)

sheets    = pd.read_excel(DATA_FILE, sheet_name=None)
req_feats = FEATURES + ['target_ratio', 'ma7d', 'Price (Rs./MWh)']

for mkt in MARKETS:
    print(f"\n{'─'*55}\nMarket: {mkt}\n{'─'*55}")

    df_full = build_features(sheets[mkt])
    df_full = df_full.dropna(subset=req_feats).copy()
    df_full = df_full[np.isfinite(df_full['target_ratio'])].copy()

    df = df_full[df_full['datetime'] >= LAST_YEAR_START].copy()
    print(f"  Rows: {len(df):,}  ({df['datetime'].min().date()} → {df['datetime'].max().date()})")

    train = df[df['datetime'] <= TRAIN_END]
    val   = df[(df['datetime'] > TRAIN_END) & (df['datetime'] <= VAL_END)]
    test  = df[df['datetime'] > VAL_END]
    print(f"  Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")
    print(f"  Price mean — Train: {train['Price (Rs./MWh)'].mean():.0f}"
          f"  Val: {val['Price (Rs./MWh)'].mean():.0f}"
          f"  Test: {test['Price (Rs./MWh)'].mean():.0f}")

    X_tr = train[FEATURES].values
    X_va = val[FEATURES].values
    X_te = test[FEATURES].values
    y_tr_ratio = train['target_ratio'].values
    y_va_ratio = val['target_ratio'].values
    ma7d_va   = val['ma7d'].values
    ma7d_te   = test['ma7d'].values
    y_va_abs  = val['Price (Rs./MWh)'].values
    y_te_abs  = test['Price (Rs./MWh)'].values

    # ── Step 1: StandardScaler ────────────────────────────────────
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)
    X_te_s = scaler.transform(X_te)

    # ── Step 2: PCA ───────────────────────────────────────────────
    pca = PCA(n_components=PCA_VAR, random_state=SEED)
    X_tr_p = pca.fit_transform(X_tr_s)
    X_va_p = pca.transform(X_va_s)
    X_te_p = pca.transform(X_te_s)
    n_comp = pca.n_components_
    var_exp = pca.explained_variance_ratio_.cumsum()[-1] * 100
    print(f"\n  PCA: {len(FEATURES)} features → {n_comp} components "
          f"({var_exp:.2f}% variance retained)")

    # PCA explained variance bar chart
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    ax.bar(range(1, n_comp+1), pca.explained_variance_ratio_ * 100,
           color='#e67e22', alpha=0.85)
    ax.plot(range(1, n_comp+1), pca.explained_variance_ratio_.cumsum() * 100,
            color='#27ae60', lw=2, marker='o', ms=3, label='Cumulative')
    ax.axhline(99.5, color='yellow', lw=1, ls='--', label='99.5% threshold')
    ax.set_title(f'PCA Explained Variance — {mkt}', color='white')
    ax.set_xlabel('Component', color='#aaa')
    ax.set_ylabel('Variance Explained (%)', color='#aaa')
    ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'{OUT}/pca_variance_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # Feature loadings heatmap (top 10 components × all features)
    n_show = min(10, n_comp)
    loadings = pd.DataFrame(
        pca.components_[:n_show],
        index=[f'PC{i+1}' for i in range(n_show)],
        columns=FEATURES)
    fig, ax = plt.subplots(figsize=(18, 5))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    im = ax.imshow(loadings.values, aspect='auto', cmap='RdBu_r', vmin=-0.4, vmax=0.4)
    ax.set_xticks(range(len(FEATURES)))
    ax.set_xticklabels(FEATURES, rotation=90, fontsize=7, color='#ccc')
    ax.set_yticks(range(n_show))
    ax.set_yticklabels(loadings.index, color='#ccc')
    ax.set_title(f'PCA Feature Loadings (Top {n_show} PCs) — {mkt}', color='white')
    plt.colorbar(im, ax=ax, label='Loading')
    plt.tight_layout()
    plt.savefig(f'{OUT}/pca_loadings_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # ── Step 3: XGBoost on PCA features ──────────────────────────
    # Feature names for XGBoost DMatrix
    pc_names = [f'PC{i+1}' for i in range(n_comp)]
    dtrain = xgb.DMatrix(X_tr_p, label=y_tr_ratio, feature_names=pc_names)
    dval   = xgb.DMatrix(X_va_p, label=y_va_ratio, feature_names=pc_names)
    dtest  = xgb.DMatrix(X_te_p, feature_names=pc_names)

    params = {
        'objective': 'reg:squarederror', 'eval_metric': 'mae',
        'max_depth': 6, 'learning_rate': 0.02,
        'subsample': 0.85, 'colsample_bytree': 0.8, 'colsample_bylevel': 0.8,
        'min_child_weight': 20, 'gamma': 0.05,
        'reg_alpha': 0.1, 'reg_lambda': 1.0,
        'tree_method': 'hist', 'max_bin': 256,
        'seed': SEED, 'verbosity': 0,
    }

    print("  Training XGBoost on PCA-transformed features...")
    evals_result = {}
    model = xgb.train(
        params, dtrain,
        num_boost_round=3000,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=80,
        evals_result=evals_result,
        verbose_eval=300,
    )
    best_iter = model.best_iteration
    print(f"  Best iteration: {best_iter}")

    val_ratio_pred  = model.predict(dval,  iteration_range=(0, best_iter))
    test_ratio_pred = model.predict(dtest, iteration_range=(0, best_iter))
    val_pred_abs    = np.clip(val_ratio_pred  * ma7d_va, 0, 20000)
    test_pred_abs   = np.clip(test_ratio_pred * ma7d_te, 0, 20000)

    print("  Results:")
    m_val  = compute_metrics(y_va_abs, val_pred_abs,  'Validation')
    m_test = compute_metrics(y_te_abs, test_pred_abs, 'Test')

    naive_mae = np.mean(np.abs(y_te_abs - test['price_lag96'].values))
    print(f"    {'Naive lag96 baseline':22s}  MAE={naive_mae:7.1f}  "
          f"(XGB beats by {(naive_mae-m_test['mae'])/naive_mae*100:.1f}%)")

    # Training curve
    fig, ax = plt.subplots(figsize=(10, 4))
    fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
    ax.plot(evals_result['train']['mae'], color='#e74c3c', lw=1, label='Train MAE')
    ax.plot(evals_result['val']['mae'],   color='#27ae60', lw=1, label='Val MAE')
    ax.axvline(best_iter, color='yellow', lw=1.5, ls='--', label=f'Best={best_iter}')
    ax.set_title(f'XGBoost+PCA Curve — {mkt}', color='white')
    ax.set_xlabel('Round', color='#aaa'); ax.set_ylabel('MAE (ratio)', color='#aaa')
    ax.legend(fontsize=9); ax.tick_params(colors='#aaa')
    plt.tight_layout()
    plt.savefig(f'{OUT}/xgb_loss_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
    plt.close()

    # PC importance
    imp = model.get_score(importance_type='gain')
    if imp:
        names = list(imp.keys()); vals = list(imp.values())
        idx = np.argsort(vals)
        fig, ax = plt.subplots(figsize=(10, 7))
        fig.patch.set_facecolor('#0f1117'); ax.set_facecolor('#1a1d27')
        ax.barh([names[i] for i in idx], [vals[i] for i in idx], color='#e67e22', alpha=0.85)
        ax.set_title(f'PC Importance (Gain) — {mkt}', color='white')
        ax.tick_params(colors='#aaa'); ax.set_xlabel('Gain', color='#aaa')
        plt.tight_layout()
        plt.savefig(f'{OUT}/xgb_pc_importance_{mkt}.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
        plt.close()

    save_pred_plot(y_te_abs, test_pred_abs, f'XGBoost+PCA — {mkt}',
                   f'{OUT}/xgb_pred_{mkt}.png')
    save_monthly_plot(test, test_pred_abs, mkt, f'{OUT}/xgb_monthly_{mkt}.png')

    # ── RESIDUAL ANALYSIS ─────────────────────────────────────────
    print(f"\n  Running residual analysis...")
    residual_analysis(test, y_te_abs, test_pred_abs, mkt, OUT)

    # Save artefacts
    model.save_model(f'{OUT}/xgb_{mkt}.json')
    with open(f'{OUT}/xgb_pipeline_{mkt}.pkl', 'wb') as f:
        pickle.dump({'scaler': scaler, 'pca': pca,
                     'features': FEATURES, 'n_pca_components': n_comp}, f)
    print(f"\n  Saved model + pipeline to {OUT}/")

print(f"\n{'='*60}")
print("Done! Check outputs_xgboost_pca/")


MODEL: XGBoost + PCA (Last Year Data)
Market: GDAM only
Period: 2025-03-01 → Feb 2026
Split:  Train≤2025-09-30  |  Val≤2025-11-30  |  Test>Val
PCA variance threshold: 0.995

───────────────────────────────────────────────────────
Market: GDAM
───────────────────────────────────────────────────────
  Rows: 35,040  (2025-03-01 → 2026-02-28)
  Train: 20,449 | Val: 5,856 | Test: 8,735
  Price mean — Train: 4657  Val: 3579  Test: 4256

  PCA: 46 features → 30 components (99.52% variance retained)
  Training XGBoost on PCA-transformed features...
[0]	train-mae:0.43947	val-mae:0.38992
[254]	train-mae:0.09259	val-mae:0.19515
  Best iteration: 174
  Results:
    Validation              MAE=  681.4  RMSE=  927.3  MAPE=29.05%  (19.0% of mean)
    Test                    MAE=  766.5  RMSE= 1021.5  MAPE=18.75%  (18.0% of mean)
    Naive lag96 baseline    MAE=  432.5  (XGB beats by -77.2%)

  Monthly breakdown:
         actual  predicted    mae
ym                               
2025-11  3679.5     3